In [1]:
import numpy as np
import pennylane as qml

In [2]:
# Circuit setup
main_qubits = 3
ancilla_wires = list(range(0, main_qubits + 2))
wires = list(range(main_qubits + 2, 2 * main_qubits + 2))
all_wires = list(range(2 * main_qubits + 2))
dev = qml.device("default.qubit", wires=2 * main_qubits + 2)

In [3]:
# Functions to set up components of block encoding
def MultiControlledZ(wires, control_values=None):
    if control_values is None:
        control_values = [0] * (len(wires) - 1)
    qml.ctrl(qml.Z(wires=wires[-1]), 
             control=wires[:-1], 
             control_values=control_values)

def R(wires):
    assert len(wires) % 2 == 1
    n = len(wires)//2
    qml.PauliX(wires=wires[0])
    MultiControlledZ(wires=wires[1:n+1]+[wires[0]])
    qml.PauliX(wires=wires[0])

def Uc(base, wires, *args, **kwargs):
    assert len(wires) % 2 == 1
    n = len(wires)//2
    if isinstance(base, qml.typing.TensorLike):
        qml.ControlledQubitUnitary(base, 
                                   control_wires=wires[n], 
                                   wires=wires[:n], 
                                   control_values=[0], 
                                   unitary_check=True)
    elif isinstance(base, qml.operation.Operator) or callable(base):
        qml.ctrl(base, control=wires[n], 
                 control_values=[0])(wires=wires[:n], *args, **kwargs)
        
def Uc_adj(base, wires, *args, **kwargs):
    assert len(wires) % 2 == 1
    n = len(wires)//2
    if isinstance(base, qml.typing.TensorLike):
        qml.adjoint(qml.ControlledQubitUnitary)(base, 
                                                control_wires=wires[n], 
                                                wires=wires[:n], 
                                                control_values=[0], 
                                                unitary_check=True)
    elif isinstance(base, qml.operation.Operator) or callable(base):
        qml.ctrl(qml.adjoint(base), 
                 control=wires[n], 
                 control_values=[0])(wires=wires[:n], *args, **kwargs)

def C(wires):
    assert len(wires) % 2 == 1
    n = len(wires)//2
    for i in range(n):
        qml.Toffoli(wires=[wires[n], wires[n+i+1], wires[i]])
        
def C_adj(wires):
    assert len(wires) % 2 == 1
    n = len(wires)//2
    for i in range(n-1, -1, -1):
        qml.Toffoli(wires=[wires[n], wires[n+i+1], wires[i]])

def W(base, wires, p, *args, **kwargs):
    assert len(wires) % 2 == 1
    n = len(wires)//2
    qml.Hadamard(wires[n])
    Uc(base, wires, *args, **kwargs)
    C(wires)
    if bool(p):
        qml.S(wires[n])
    qml.Hadamard(wires[n])
    
def W_adj(base, wires, p, *args, **kwargs):
    assert len(wires) % 2 == 1
    n = len(wires)//2
    qml.Hadamard(wires[n])
    if bool(p):
        qml.adjoint(qml.S)(wires[n])
    C_adj(wires)
    Uc_adj(base, wires, *args, **kwargs)
    qml.Hadamard(wires[n])

def G(base, wires, p, *args, **kwargs):
    assert len(wires) % 2 == 1
    n = len(wires)//2
    qml.PauliZ(wires[n])
    W_adj(base, wires, p, *args, **kwargs)
    R(wires)
    W(base, wires, p, *args, **kwargs)
    
def G_adj(base, wires, p, *args, **kwargs):
    assert len(wires) % 2 == 1
    n = len(wires)//2
    W_adj(base, wires, p, *args, **kwargs)
    R(wires)
    W(base, wires, p, *args, **kwargs)
    qml.PauliZ(wires[n])

In [4]:
# Initialize block encoding function
def RealDiagonalBlockEncoding(U, wires, ancilla_wires, p=0, *args, **kwargs):
    assert len(ancilla_wires) == len(wires) + 2
    qml.Hadamard(wires=ancilla_wires[0])
    W(base=U, 
        wires=ancilla_wires[1:]+wires, 
        p=p, *args, **kwargs)
    qml.ctrl(G, control=ancilla_wires[0], 
                control_values=[0])(base=U, wires=ancilla_wires[1:]+wires,
                                    p=p, **kwargs)
    qml.ctrl(G_adj, control=ancilla_wires[0], 
                control_values=[1])(base=U, wires=ancilla_wires[1:]+wires,
                                    p=p, *args, **kwargs)
    qml.Hadamard(wires=ancilla_wires[0])
    W_adj(base=U, wires=ancilla_wires[1:]+wires, p=p, *args, **kwargs)
    qml.PauliX(wires=ancilla_wires[0])
    qml.PauliZ(wires=ancilla_wires[0])
    qml.PauliX(wires=ancilla_wires[0])

In [5]:
# Initialize block encoding
feature_vector = [1, 0, 1]
block_encoding = qml.prod(RealDiagonalBlockEncoding)(
    qml.BasisEmbedding, wires=wires, 
    ancilla_wires=ancilla_wires, 
    features=feature_vector)

In [6]:
# Set up QSVT angles
poly = np.array([0, -1, 0, 0, 0, 0.5])
angles = qml.poly_to_angles(poly, "QSVT", angle_solver="root-finding")

In [7]:
# Create projectors for QSVT
projectors = [
    qml.PCPhase(angles[i], dim=2**len(wires), wires=all_wires)
    for i in range(len(angles))
]

In [8]:
# NLAT
@qml.qnode(dev)
def circuit():
    qml.QSVT(block_encoding, projectors)
    return qml.state()

In [9]:
# Check diagonal
np.diag(qml.matrix(circuit)())[:8]

array([-1.86752407e-36-2.57869885e-34j, -1.86752407e-36-2.57869885e-34j,
       -1.86752407e-36-2.57869885e-34j, -1.86752407e-36-2.57869885e-34j,
       -1.86752407e-36-2.57869885e-34j, -5.00000000e-01+8.66025404e-01j,
       -1.86752407e-36-2.57869885e-34j, -1.86752407e-36-2.57869885e-34j])